In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import requests

In [2]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [3]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

Model='llama3.1'

In [4]:
links = fetch_website_links("https://nytimes.com")
links

['#site-content',
 '#site-index',
 '#after-dfp-ad-top',
 '/',
 '/',
 '/international/',
 '/ca/',
 'https://www.nytimes.com/es/',
 'https://cn.nytimes.com',
 'https://www.nytimes.com/section/todayspaper',
 '/',
 'https://www.nytimes.com/section/us',
 'https://www.nytimes.com/section/us',
 'https://www.nytimes.com/section/politics',
 'https://www.nytimes.com/section/nyregion',
 'https://www.nytimes.com/spotlight/california-news',
 'https://www.nytimes.com/section/education',
 'https://www.nytimes.com/section/health',
 'https://www.nytimes.com/section/obituaries',
 'https://www.nytimes.com/section/science',
 'https://www.nytimes.com/section/climate',
 'https://www.nytimes.com/section/weather',
 'https://www.nytimes.com/section/sports',
 'https://www.nytimes.com/section/business',
 'https://www.nytimes.com/section/technology',
 'https://www.nytimes.com/section/upshot',
 'https://www.nytimes.com/section/magazine',
 'https://www.nytimes.com/spotlight/donald-trump',
 'https://www.nytimes.com/

In [5]:
#Use lamma3.1 to read the link son the webpage and exrtract usefull links in json format

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

#user_prompt += "text"
#user_prompt = user_prompt + "text"

In [7]:
print(get_links_user_prompt("https://nytimes.com"))


Here is the list of links on the website https://nytimes.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#site-content
#site-index
#after-dfp-ad-top
/
/
/international/
/ca/
https://www.nytimes.com/es/
https://cn.nytimes.com
https://www.nytimes.com/section/todayspaper
/
https://www.nytimes.com/section/us
https://www.nytimes.com/section/us
https://www.nytimes.com/section/politics
https://www.nytimes.com/section/nyregion
https://www.nytimes.com/spotlight/california-news
https://www.nytimes.com/section/education
https://www.nytimes.com/section/health
https://www.nytimes.com/section/obituaries
https://www.nytimes.com/section/science
https://www.nytimes.com/section/climate
https://www.nytimes.com/section/weather
https://www.nytimes.com/section/sports
https://www.nytimes.com/section/business
https://www.ny

In [8]:
def select_relevant_links(url):
    response =  ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)          # strings to dict
    return links
    

In [9]:
select_relevant_links("https://nytimes.com")

{'title': 'New York Times',
 'url': 'https://www.nytimes.com/',
 'pages': 'at least 100'}

In [10]:
select_relevant_links("https://cricbuzz.com")

{'Cricket News': [{'title': 'London Spirit secure Brevis, Zampa in eight direct signings',
   'link': 'https://www.cricbuzz.com/en/news/articles/137352=london-spirit-secure-brevis-zampa-in-eight-direct-signings'},
  {'title': 'All confirmed signings ahead of the Hundred 2026',
   'link': 'https://www.cricbuzz.com/en/news/articles/137264=all-confirmed-signings-ahead-of-the-hundred-2026'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {Model}")
    response = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(links)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://cricbuzz.com")

Selecting relevant links for https://cricbuzz.com by calling llama3.1
{'links': [{'text': 'cricket schedule', 'url': '/schedule/upcoming-series/international'}, {'text': 'cricket videos', 'url': '/videos/'}, {'text': 'cricket news', 'url': '/news/'}, {'text': 'Times of India', 'url': 'https://timesofindia.indiatimes.com/'}, {'text': 'Navbharat Times', 'url': 'https://navbharattimes.indiatimes.com//'}], 'apps': [{'type': 'Google Play', 'store_url': 'https://play.google.com/store/apps/details?id=com.cricbuzz.android'}, {'type': 'App Store (iOS)', 'store_url': 'https://itunes.apple.com/app/id360466413'}, {'text': 'Facebook Page', 'url': 'https://www.facebook.com/cricbuzz'}, {'text': 'Twitter Handle', 'url': 'https://twitter.com/cricbuzz'}, {'type': 'YouTube Channel', 'channel_url': 'https://www.youtube.com/c/cricbuzz'}, {'type': 'Pinterest Page', 'page_url': 'https://in.pinterest.com/cricbuzz/'}], 'urls': ['https://product-blog.cricbuzz.com/cricbuzz-mobile-apps-tv-ad-cricket-ka-keeda/', '

{'links': [{'text': 'cricket schedule',
   'url': '/schedule/upcoming-series/international'},
  {'text': 'cricket videos', 'url': '/videos/'},
  {'text': 'cricket news', 'url': '/news/'},
  {'text': 'Times of India', 'url': 'https://timesofindia.indiatimes.com/'},
  {'text': 'Navbharat Times',
   'url': 'https://navbharattimes.indiatimes.com//'}],
 'apps': [{'type': 'Google Play',
   'store_url': 'https://play.google.com/store/apps/details?id=com.cricbuzz.android'},
  {'type': 'App Store (iOS)',
   'store_url': 'https://itunes.apple.com/app/id360466413'},
  {'text': 'Facebook Page', 'url': 'https://www.facebook.com/cricbuzz'},
  {'text': 'Twitter Handle', 'url': 'https://twitter.com/cricbuzz'},
  {'type': 'YouTube Channel',
   'channel_url': 'https://www.youtube.com/c/cricbuzz'},
  {'type': 'Pinterest Page',
   'page_url': 'https://in.pinterest.com/cricbuzz/'}],
 'urls': ['https://product-blog.cricbuzz.com/cricbuzz-mobile-apps-tv-ad-cricket-ka-keeda/',
  'https://in.pinterest.com/cricb

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'models page', 'url': '/models'}, {'type': 'datasets page', 'url': '/datasets'}, {'type': 'spaces page', 'url': '/spaces'}, {'type': 'docs page', 'url': '/docs'}, {'type': 'about company', 'url': '/huggingface'}, {'type': 'branding', 'url': '/brand'}, {'type': 'alliances', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'blog', 'url': 'https://discuss.huggingface.co'}, {'type': 'status updates', 'url': 'https://status.huggingface.co/'}, {'type': 'models', 'url': '/inference/models'}, {'type': 'pricing page', 'url': '/pricing#endpoints'}, {'type': 'model endpoints', 'url': '/docs/transformers.js'}]}
Found 12 relevant links


{'links': [{'type': 'models page', 'url': '/models'},
  {'type': 'datasets page', 'url': '/datasets'},
  {'type': 'spaces page', 'url': '/spaces'},
  {'type': 'docs page', 'url': '/docs'},
  {'type': 'about company', 'url': '/huggingface'},
  {'type': 'branding', 'url': '/brand'},
  {'type': 'alliances',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'blog', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status updates', 'url': 'https://status.huggingface.co/'},
  {'type': 'models', 'url': '/inference/models'},
  {'type': 'pricing page', 'url': '/pricing#endpoints'},
  {'type': 'model endpoints', 'url': '/docs/transformers.js'}]}

In [14]:
#Assemble all the details into another prompt to ollama
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'About page', 'url': 'https://huggingface.co/'}], 'https://huggingface.co/models': {'type': 'models page', 'url': 'https://huggingface.co/models'}, '{"type": "careers page","url": "https://apply.workable.com/huggingface/"}': {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, 'https://huggingface.co/pricing': {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, 'https://www.linkedin.com/company/huggingface/': {'type': 'linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}, 'https://twitter.com/huggingface': {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, 'https://github.com/huggingface': {'type': 'github page', 'url': 'https://github.com/huggingface'}, 'https://discuss.huggingface.co': {'type': 'forums'}, 'https://status.huggingface.co/': {'type': 'status', 'url': 'https://status.huggingface.co/'}, 'https://huggingface.c

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customersand careers/jobs if you have the information.
"""

# # Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'company page', 'url': 'https://huggingface.co'}, {'type': 'about us page', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'teams page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'blog page', 'url': 'https://discuss.huggingface.co'}, {'type': 'forums for users or a community page', 'url': 'https://status.huggingface.co/'}, {'type': 'social media link to show support', 'url': 'https://twitter.com/huggingface'}, {'type': 'source code repository page (GitHub)', 'url': 'https://github.com/huggingface'}]}
Found 8 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-Image\nUpdated\n4 days ago\n•\n7.59k\n•\n844\nLightricks/LTX-2\nUpdated\nabout 18 hours ago\n•\n1.54M\n•\n1.16k\nfal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA\nUpdated\n12 days ago\n•\n55.2k\n•\n756\nopenbmb/AgentCPM-Explore\nUpdated\nabout 21 hours ago\n•\n1.83k\n•\n351\ngoogle/translategemma-4b-it\nUpdated\n4 days ago\n•\n17.6k\n•\n322\nBrowse 2M+ models\nSpaces\nRunning\non\n

In [21]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'About page', 'url': 'https://huggingface.co/brand'}, {'type': 'Careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Jobs page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Company website', 'url': 'https://huggingface.co'}, {'type': 'Blog', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Twitter account', 'url': 'https://twitter.com/huggingface'}]}
Found 8 relevant links


**Welcome to Hugging Face**

Hugging Face is a leading platform for the machine learning community, where they collaborate on models, datasets, and applications. Our mission is to empower the next generation of machine learning engineers, scientists, and users to learn, collaborate, and share their work.

**Our Community**

With over 1 million members, our community is at the forefront of AI research and innovation. Our Hub allows anyone to share, explore, discover, and experiment with open-source ML models and datasets.

**Core Products**

* **Models**: Browse 2M+ open-source models for text, image, video, audio, or 3D applications.
* **Datasets**: Explore 500k+ datasets and collections of data used in machine learning research.
* **Spaces**: Run, deploy, and share AI models with Spaces, our development and deployment platform.

**About Us**

Hugging Face is founded on the principles of open-source collaboration and community building. Our team includes experienced technologists from top organizations around the world. We believe in the power of collective knowledge and innovation to create an inclusive and accessible future for machine learning.

**Careers with Hugging Face**

We're always looking for talented professionals who share our passion for AI innovation. Join our team to work on cutting-edge projects, collaborate with experts, and shape the future of ML and NLP.

Contact us today to learn more about joining our community and working together to build sustainable and responsible technologies that benefit society as a whole.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True  # streanms one by one in chunks (parts)
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.1
{'links': [{'type': 'Company page', 'url': 'https://huggingface.co'}, {'type': 'Careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'About page (Brand)', 'url': 'https://huggingface.co/brand'}, {'type': 'Documentation (Transformers)', 'url': 'https://huggingface.co/docs/transformers'}, {'type': 'Documentation (Diffusers)', 'url': 'https://huggingface.co/docs/diffusers'}, {'type': 'Documentation (Safetensors)', 'url': 'https://huggingface.co/docs/safetensors'}, {'type': 'Inference Endpoints', 'url': 'https://endpoints.huggingface.co'}, {'type': 'Contact page', 'url': 'https://twitter.com/huggingface'}]}
Found 8 relevant links


**Hugging Face: Empowering Machine Learning Collaboration**

At Hugging Face, we're redefining the way machine learning is developed, shared, and adopted. As a community-driven platform, we provide the tools and resources for ML developers to collaborate, innovate, and push the boundaries of what's possible.

**Our Mission**
Hugging Face is dedicated to creating an open and inclusive AI future by providing a hub where anyone can share, explore, discover, and experiment with open-source machine learning. We empower the next generation of ML engineers, scientists, and end users to learn, collaborate, and share their work towards building an open and ethical AI future.

**Key Features**

* **Hugging Face Hub**: A central platform for sharing, exploring, discovering, and experimenting with open-source ML models, datasets, and applications.
* **20M+ models and 500k+ datasets**: Our vast repository of pre-trained models and datasets accelerates your ML projects.
* **Collaboration features**: Host and collaborate on unlimited public models, datasets, and applications with our seamless platform.
* **Compute and Environment (En)**: Enjoy paid Compute and En for accelerated development.

**Join the Movement**

Hugging Face welcomes both individuals and organizations to join its community. If you're passionate about machine learning and want to contribute to shaping the future of AI, we invite you to:

* [Sign Up](#) to start collaborating on ML projects today
* Learn more about our **Enterprise solutions** for custom deployment and integration

**Building an Inclusive Future**

At Hugging Face, we're committed to creating a culture where everyone can grow and contribute. Our values of inclusivity, diversity, and collaboration drive us towards building an equitable AI future.

### Career Opportunities

If you'd like to join our team and build innovative technologies with a talented group of developers, data scientists, and engineers, we invite you to explore our current job openings at [Hugging Face - Current Openings](#)

**How We're Making a Difference**

Our work is centered around three core areas:

* **Openness**: We foster an open-source ecosystem where everyone can share and innovate.
* **Collaboration**: Our platform encourages collaboration, transparency, and shared knowledge among the ML community.

Join us today to be part of shaping the AI landscape.

In [ ]:
#Humorous system prompt
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
stream_brochure("CricBuzz", "https://nationalgeographic.com")

In [ ]:
stream_brochure("VU pune", "https://vupune.ac.in")